# T5Gemma2 PubMed drafter retraining on Google Colab

This run uses the base `google/t5gemma-2-1b-1b` target on an L4 Colab VM. It first checks native-versus-cached greedy parity and block-versus-serial margin safety. Near-tied blocks and rejected proposals fall back to serial target decoding so the committed cache follows the native path; a wrong-proposal and oracle-proposal test must both match before training starts. Only if that gate passes does it collect 1,000 target-greedy training traces, train the drafter for 5,000 updates, and benchmark against native generation on 128 PubMed validation examples.

The training objective is a sigmoid top-1 logit-margin accepted-length surrogate plus a small CE term after 1,000 CE warm-up updates. It is an adaptation for deterministic `argmax` proposals, not the paper's probability-mass EAL loss. No PubMed test split is used. T5Gemma2 weights are downloaded only inside the Colab VM; the output bundle contains the drafter checkpoint and reports, not target weights.


In [ ]:
# @title 1. Confirm runtime and run limits
import json
import os
import random
import shutil
import subprocess
import sys
import time
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch
from tqdm.auto import tqdm

SEED = 42
TARGET_MODEL_ID = "google/t5gemma-2-1b-1b"
TARGET_REVISION = "dd0a2683227859151b1730ca3a63087df5b5f39b"
DATASET_ID = "ccdv/pubmed-summarization"
DATASET_CONFIG = "document"
TRAIN_LIMIT = 1000
VALIDATION_LIMIT = 128
MAX_SOURCE_TOKENS = 4096
MAX_NEW_TOKENS = 512
TRAIN_STEPS = 5000
MARGIN_WARMUP_STEPS = 1000
BLOCK_SIZE = 4
BENCHMARK_REPEATS = 3
BENCHMARK_WARMUP = 4

if not torch.cuda.is_available():
    raise RuntimeError("No Colab GPU is attached. Choose Runtime > Change runtime type > GPU.")
GPU_NAME = torch.cuda.get_device_name(0)
if "L4" not in GPU_NAME:
    raise RuntimeError(f"This run is set up for L4, but Colab assigned {GPU_NAME!r}.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("L4 did not report BF16 support; stopping instead of silently changing precision.")

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
RUN_ROOT = Path("/content/t5gemma2_margin_retrain")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RUN_DIR = RUN_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
HF_HOME = Path("/content/t5gemma2_model_cache")
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print(json.dumps({
    "gpu": GPU_NAME,
    "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    "torch": torch.__version__,
    "bf16": True,
    "run_dir": str(RUN_DIR),
    "train_examples": TRAIN_LIMIT,
    "validation_examples": VALIDATION_LIMIT,
    "source_limit": MAX_SOURCE_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "drafter_steps": TRAIN_STEPS,
}, indent=2), flush=True)


In [ ]:
# @title 2. Install runtime dependencies and stage the current source package
import importlib.metadata

PACKAGE_ARCHIVE = Path("/content/encdec_drafter_src.zip")
if not PACKAGE_ARCHIVE.is_file():
    raise FileNotFoundError("Upload the current encdec_drafter_src.zip to /content before running this cell.")

required = {
    "transformers": "5.14.1",
    "tokenizers": "0.22.2",
}
def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

needs_install = any(package_version(name) != version for name, version in required.items())
needs_install = needs_install or package_version("datasets") is None or package_version("sentencepiece") is None or package_version("PyYAML") is None
if needs_install:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "transformers==5.14.1", "tokenizers==0.22.2", "datasets>=3,<6",
        "sentencepiece", "pyyaml", "accelerate",
    ], check=True)

PACKAGE_ROOT = RUN_DIR / "package"
PACKAGE_ROOT.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(PACKAGE_ARCHIVE) as archive:
    archive.extractall(PACKAGE_ROOT)
if not (PACKAGE_ROOT / "encdec_drafter" / "__init__.py").is_file():
    raise RuntimeError("Source zip must contain encdec_drafter/__init__.py at its root.")
sys.path.insert(0, str(PACKAGE_ROOT))

from encdec_drafter.config import load_config
from encdec_drafter.data import collect_teacher_traces, read_jsonl, source_text
from encdec_drafter.diagnostics import diagnose_t5gemma2_verifier
from encdec_drafter.model import load_adapter
from encdec_drafter.training import train_drafter

print("transformers", package_version("transformers"), "tokenizers", package_version("tokenizers"), flush=True)
print("package", PACKAGE_ROOT, flush=True)


In [ ]:
# @title 3. Download the gated target weights to Colab only
from huggingface_hub import snapshot_download

CACHED_MODEL_PATH = HF_HOME / "hub" / "models--google--t5gemma-2-1b-1b" / "snapshots" / TARGET_REVISION
if (CACHED_MODEL_PATH / "config.json").is_file():
    MODEL_PATH = CACHED_MODEL_PATH.resolve()
    print("Reusing model cache already on Colab:", MODEL_PATH, flush=True)
else:
    TOKEN_FILE = Path("/content/.hf_token_runtime")
    if not TOKEN_FILE.is_file():
        raise RuntimeError("The one-use HF token file is missing. No model weights were downloaded.")
    hf_token = TOKEN_FILE.read_text(encoding="utf-8").strip()
    TOKEN_FILE.unlink()
    if not hf_token:
        raise RuntimeError("The one-use HF token file was empty.")
    MODEL_PATH = Path(snapshot_download(
        repo_id=TARGET_MODEL_ID,
        revision=TARGET_REVISION,
        token=hf_token,
        cache_dir=str(HF_HOME / "hub"),
    )).resolve()
    del hf_token
    print("Model cached on Colab:", MODEL_PATH, flush=True)
print("Pinned revision:", MODEL_PATH.name, flush=True)


In [ ]:
# @title 4. Stream deterministic PubMed train and validation samples
from datasets import load_dataset

train_stream = load_dataset(DATASET_ID, DATASET_CONFIG, split="train", streaming=True).shuffle(
    seed=SEED, buffer_size=10000
)
validation_stream = load_dataset(DATASET_ID, DATASET_CONFIG, split="validation", streaming=True)

def select_rows(stream, wanted, split_name):
    selected = []
    for row in tqdm(stream, desc=f"Read {split_name}"):
        article = row.get("article")
        abstract = row.get("abstract")
        if isinstance(article, str) and article.strip() and isinstance(abstract, str) and abstract.strip():
            selected.append({
                "id": str(row.get("id") or f"{split_name}-{len(selected)}"),
                "text": article.strip(),
                "summary": abstract.strip(),
            })
        if len(selected) >= wanted:
            break
    if len(selected) != wanted:
        raise RuntimeError(f"Expected {wanted} usable {split_name} rows, got {len(selected)}.")
    return selected

train_rows = select_rows(train_stream, TRAIN_LIMIT, "train")
validation_rows = select_rows(validation_stream, VALIDATION_LIMIT, "validation")
TRAIN_FILE = RUN_DIR / "train.jsonl"
VALIDATION_FILE = RUN_DIR / "validation.jsonl"

def write_rows(path, rows):
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

write_rows(TRAIN_FILE, train_rows)
write_rows(VALIDATION_FILE, validation_rows)
print(json.dumps({
    "dataset": DATASET_ID,
    "config": DATASET_CONFIG,
    "train_examples": len(train_rows),
    "validation_examples": len(validation_rows),
    "test_split_used": False,
    "first_train_id": train_rows[0]["id"],
    "first_validation_id": validation_rows[0]["id"],
}, indent=2), flush=True)
del train_stream, validation_stream, train_rows, validation_rows


In [ ]:
# @title 5. Build config, load T5Gemma2, and gate verifier correctness
SOURCE_PREFIX = (
    "Summarize the following biomedical research article into a concise, factual abstract. "
    "Preserve the key objective, methods, results, and conclusion; do not add information.\nArticle:\n"
)
TRACE_FILE = RUN_DIR / "teacher_traces.jsonl"
DRAFTER_FILE = RUN_DIR / "drafter.pt"
CONFIG_PATH = RUN_DIR / "runtime.yaml"
BENCH_DIR = RUN_DIR / "benchmark"
config = {
    "target": {
        "checkpoint": str(MODEL_PATH),
        "identity": f"{TARGET_MODEL_ID}@{TARGET_REVISION}",
        "tokenizer": None,
        "local_files_only": True,
        "trust_remote_code": False,
        "device": "cuda",
        "dtype": "bfloat16",
        "attn_implementation": None,
    },
    "data": {
        "training_file": str(TRAIN_FILE),
        "validation_file": str(VALIDATION_FILE),
        "test_file": str(VALIDATION_FILE),
        "id_field": "id",
        "source_field": "text",
        "target_field": "summary",
        "detokenize": True,
        "source_prefix": SOURCE_PREFIX,
        "max_source_tokens": MAX_SOURCE_TOKENS,
    },
    "generation": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "min_new_tokens": 32,
        "repetition_penalty": 1.05,
        "no_repeat_ngram_size": 3,
        "temperature": 0.0,
        "top_k": 0,
        "top_p": 1.0,
        "num_beams": 1,
    },
    "draft": {
        "width": 256,
        "layers": 2,
        "heads": 4,
        "ffn_width": 1024,
        "max_block_size": 8,
        "target_feature_window": 64,
        "max_source_tokens": MAX_SOURCE_TOKENS,
        "checkpoint": None,
    },
    "source_memory": {"mode": "full", "budget": 256},
    "speculation": {"block_size": BLOCK_SIZE, "ambiguity_fallback_margin": 0.5},
    "training": {
        "trace_file": str(TRACE_FILE),
        "output_checkpoint": str(DRAFTER_FILE),
        "steps": TRAIN_STEPS,
        "learning_rate": 0.0003,
        "weight_decay": 0.01,
        "seed": SEED,
        "log_every": 10,
        "save_every": 500,
        "max_grad_norm": 1.0,
        "loss_type": "greedy_margin_accepted_length",
        "margin_warmup_steps": MARGIN_WARMUP_STEPS,
        "margin_ce_weight": 0.05,
        "margin_temperature": 1.0,
    },
    "benchmark": {"output_dir": str(BENCH_DIR), "warmup": BENCHMARK_WARMUP, "repeats": BENCHMARK_REPEATS},
}
import yaml
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
loaded_config = load_config(CONFIG_PATH)
adapter = load_adapter(loaded_config)
print("target", json.dumps(adapter.target_manifest(), ensure_ascii=False), flush=True)

GATE_FILE = RUN_DIR / "verifier_gate.json"
gate = diagnose_t5gemma2_verifier(
    adapter,
    loaded_config,
    VALIDATION_FILE,
    limit=8,
    prefix_tokens=32,
    block_sizes=(2, 4, 8),
    output_path=GATE_FILE,
)
print("verifier_gate", json.dumps({key: gate[key] for key in (
    "status", "target_only_exact_all_examples", "full_prefix_vs_serial_top1_exact_all_examples",
    "block_vs_serial_top1_exact_all_examples", "decision",
)}, ensure_ascii=False), flush=True)
if gate["status"] != "passed":
    raise RuntimeError("Verifier gate failed; skipped trace collection and drafter training to avoid wasted compute.")


In [ ]:
if "gate" not in globals() or gate.get("status") != "passed":
    raise RuntimeError("Verifier gate did not pass; refusing to collect traces or train.")

# @title 6. Collect teacher traces and train the drafter
start_time = time.perf_counter()
trace_result = collect_teacher_traces(
    adapter, loaded_config, TRAIN_FILE, TRACE_FILE, limit=TRAIN_LIMIT, overwrite=True
)
print("teacher_traces", json.dumps(trace_result, ensure_ascii=False), flush=True)
training_result = train_drafter(
    adapter,
    loaded_config,
    TRACE_FILE,
    output_path=DRAFTER_FILE,
    max_steps=TRAIN_STEPS,
)
TRAIN_RESULT_FILE = RUN_DIR / "training_result.json"
TRAIN_RESULT_FILE.write_text(json.dumps(training_result, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("training", json.dumps(training_result, ensure_ascii=False), flush=True)
print("training_elapsed_seconds", round(time.perf_counter() - start_time, 2), flush=True)


In [ ]:
if "training_result" not in globals():
    raise RuntimeError("Training cell did not complete; refusing to benchmark or package results.")

# @title 7. Benchmark exactness and latency; package results
from encdec_drafter.benchmark import benchmark_runtime

benchmark_result = benchmark_runtime(
    adapter,
    loaded_config,
    VALIDATION_FILE,
    limit=VALIDATION_LIMIT,
    repeats=BENCHMARK_REPEATS,
    warmup=BENCHMARK_WARMUP,
    draft_checkpoint=DRAFTER_FILE,
    output_dir=BENCH_DIR,
)
benchmark_run = Path(benchmark_result["run_dir"])
benchmark_summary = json.loads((benchmark_run / "summary.json").read_text(encoding="utf-8"))
exact_rate = float(benchmark_summary["exact_token_match_rate"])
median_speedup = float(benchmark_summary["per_request_speedup"]["median"])
if exact_rate < 1.0:
    decision = "STOP: exact greedy parity failed; do not interpret latency numbers as a valid speedup."
elif median_speedup <= 1.0:
    decision = "STOP: parity passed, but this run did not beat native cached generation on median latency."
else:
    decision = "Positive validation result; repeat and widen evaluation before making a speedup claim."

PILOT_SUMMARY = {
    "status": "completed",
    "target_model": TARGET_MODEL_ID,
    "target_revision": TARGET_REVISION,
    "gpu": GPU_NAME,
    "dataset": f"{DATASET_ID}:{DATASET_CONFIG}",
    "train_examples": TRAIN_LIMIT,
    "validation_examples": VALIDATION_LIMIT,
    "test_split_used": False,
    "source_token_cap": MAX_SOURCE_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "drafter_steps": training_result["steps"],
    "loss": loaded_config["training"]["loss_type"],
    "margin_warmup_steps": MARGIN_WARMUP_STEPS,
    "margin_ce_weight": loaded_config["training"]["margin_ce_weight"],
    "block_size": BLOCK_SIZE,
    "verifier_gate": gate,
    "exact_token_match_rate": exact_rate,
    "median_per_request_speedup": median_speedup,
    "native_completion_seconds": benchmark_summary["native_completion_seconds"],
    "speculative_completion_seconds": benchmark_summary["speculative_completion_seconds"],
    "per_request_speedup": benchmark_summary["per_request_speedup"],
    "native_peak_allocated_bytes": benchmark_summary.get("native_peak_allocated_bytes"),
    "speculative_peak_allocated_bytes": benchmark_summary.get("speculative_peak_allocated_bytes"),
    "decision": decision,
    "draft_checkpoint": str(DRAFTER_FILE),
    "benchmark_dir": str(benchmark_run),
    "elapsed_seconds": round(time.perf_counter() - start_time, 2),
    "not_a_paper_result": True,
}
SUMMARY_FILE = RUN_DIR / "pilot_summary.json"
SUMMARY_FILE.write_text(json.dumps(PILOT_SUMMARY, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
RESULTS_BUNDLE = RUN_DIR / "colab_results.zip"
with zipfile.ZipFile(RESULTS_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (
        SUMMARY_FILE,
        GATE_FILE,
        TRAIN_RESULT_FILE,
        CONFIG_PATH,
        DRAFTER_FILE,
        benchmark_run / "summary.json",
        benchmark_run / "requests.jsonl",
        benchmark_run / "manifest.json",
    ):
        archive.write(path, path.relative_to(RUN_DIR))
print(json.dumps(PILOT_SUMMARY, indent=2, ensure_ascii=False), flush=True)
print("PILOT_SUMMARY_PATH=" + str(SUMMARY_FILE), flush=True)
print("RESULTS_BUNDLE_PATH=" + str(RESULTS_BUNDLE), flush=True)
